In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from getpass import getpass
from typing import Any

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
SOURCE_COLLECTION_NAME = "SnapshotArticle"

if client.collections.exists(SOURCE_COLLECTION_NAME):
    client.collections.delete(SOURCE_COLLECTION_NAME)

articles = client.collections.create(
    name=SOURCE_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="external_id",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="version",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="published",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", SOURCE_COLLECTION_NAME)

Created collection: SnapshotArticle


In [5]:
snapshot_articles = [
    {
        "external_id": "snap-001",
        "title": "Weaviate Cloud Authentication",
        "content": "Weaviate Cloud uses a cluster URL and a Weaviate API key. Provider headers such as the OpenAI API key are used for vectorization and generation.",
        "topic": "cloud",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
    {
        "external_id": "snap-002",
        "title": "Vector Search",
        "content": "Vector search finds semantically similar objects by comparing embeddings in vector space.",
        "topic": "vector_search",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
    {
        "external_id": "snap-003",
        "title": "BM25 Keyword Search",
        "content": "BM25 is a keyword-based search method useful for exact terms, technical names and categories.",
        "topic": "bm25",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
    {
        "external_id": "snap-004",
        "title": "Hybrid Search",
        "content": "Hybrid search combines vector search and BM25. The alpha parameter controls the balance between semantic and keyword ranking.",
        "topic": "hybrid_search",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
    {
        "external_id": "snap-005",
        "title": "Generative Search",
        "content": "Generative search retrieves relevant objects and sends them to a language model to produce an answer.",
        "topic": "generative_search",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
    {
        "external_id": "snap-006",
        "title": "Focused RAG",
        "content": "Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown concept notes.",
        "topic": "rag",
        "source": "markdown",
        "version": 2,
        "published": True,
    },
    {
        "external_id": "snap-007",
        "title": "Named Vectors",
        "content": "Named vectors allow one object to have multiple vector representations such as title_vector and description_vector.",
        "topic": "named_vectors",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
    {
        "external_id": "snap-008",
        "title": "Cross-References",
        "content": "Cross-references connect objects between collections using reference properties.",
        "topic": "cross_references",
        "source": "notebook",
        "version": 1,
        "published": False,
    },
    {
        "external_id": "snap-009",
        "title": "Multi-Tenancy",
        "content": "Multi-tenancy isolates data for different customers, users or organizations inside one shared collection schema.",
        "topic": "multi_tenancy",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
    {
        "external_id": "snap-010",
        "title": "Idempotent Imports",
        "content": "Deterministic UUIDs and upsert logic help avoid duplicate objects during repeated imports.",
        "topic": "data_ingestion",
        "source": "notebook",
        "version": 1,
        "published": True,
    },
]

In [6]:
result = articles.data.insert_many(snapshot_articles)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = articles.query.fetch_objects(
    limit=20,
    return_properties=[
        "external_id",
        "title",
        "topic",
        "source",
        "version",
        "published",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print(obj.properties)
    print("-" * 80)

UUID: 0e9010f9-997e-4441-be5e-aec93a3331a0
{'source': 'notebook', 'published': True, 'topic': 'bm25', 'external_id': 'snap-003', 'title': 'BM25 Keyword Search', 'version': 1}
--------------------------------------------------------------------------------
UUID: 1ed6c91e-2c3a-4eb7-9181-b210c3f116ea
{'source': 'notebook', 'published': True, 'topic': 'generative_search', 'external_id': 'snap-005', 'title': 'Generative Search', 'version': 1}
--------------------------------------------------------------------------------
UUID: 31948174-9e8e-45cc-8fe2-5760d7097f86
{'source': 'notebook', 'published': True, 'topic': 'vector_search', 'external_id': 'snap-002', 'title': 'Vector Search', 'version': 1}
--------------------------------------------------------------------------------
UUID: 52e34125-57bc-4908-b4b6-2d1ef1049bf0
{'source': 'markdown', 'published': True, 'topic': 'rag', 'external_id': 'snap-006', 'title': 'Focused RAG', 'version': 2}
----------------------------------------------------

In [8]:
SNAPSHOT_DIR = Path("vector_db/snapshots")
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOT_PATH = SNAPSHOT_DIR / "snapshot_articles.jsonl"

print(SNAPSHOT_PATH)

vector_db/snapshots/snapshot_articles.jsonl


In [9]:
def export_collection_to_jsonl(
    collection,
    output_path: Path,
    *,
    return_properties: list[str],
) -> dict[str, Any]:
    exported_count = 0

    with output_path.open("w", encoding="utf-8") as file:
        for obj in collection.iterator(
            return_properties=return_properties,
        ):
            record = {
                "uuid": str(obj.uuid),
                "properties": obj.properties,
            }

            file.write(json.dumps(record, ensure_ascii=False) + "\n")
            exported_count += 1

    return {
        "output_path": str(output_path),
        "exported_count": exported_count,
    }

In [10]:
export_report = export_collection_to_jsonl(
    articles,
    SNAPSHOT_PATH,
    return_properties=[
        "external_id",
        "title",
        "content",
        "topic",
        "source",
        "version",
        "published",
    ],
)

export_report

{'output_path': 'vector_db/snapshots/snapshot_articles.jsonl',
 'exported_count': 10}

In [11]:
with SNAPSHOT_PATH.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(line.strip())

        if line_number >= 3:
            break

{"uuid": "0e9010f9-997e-4441-be5e-aec93a3331a0", "properties": {"source": "notebook", "content": "BM25 is a keyword-based search method useful for exact terms, technical names and categories.", "published": true, "topic": "bm25", "external_id": "snap-003", "title": "BM25 Keyword Search", "version": 1}}
{"uuid": "1ed6c91e-2c3a-4eb7-9181-b210c3f116ea", "properties": {"source": "notebook", "content": "Generative search retrieves relevant objects and sends them to a language model to produce an answer.", "published": true, "topic": "generative_search", "external_id": "snap-005", "title": "Generative Search", "version": 1}}
{"uuid": "31948174-9e8e-45cc-8fe2-5760d7097f86", "properties": {"source": "notebook", "title": "Vector Search", "published": true, "topic": "vector_search", "external_id": "snap-002", "content": "Vector search finds semantically similar objects by comparing embeddings in vector space.", "version": 1}}


In [12]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

    return records

In [13]:
snapshot_records = load_jsonl(SNAPSHOT_PATH)

print("Loaded records:", len(snapshot_records))

for record in snapshot_records[:3]:
    print(record)
    print("-" * 80)

Loaded records: 10
{'uuid': '0e9010f9-997e-4441-be5e-aec93a3331a0', 'properties': {'source': 'notebook', 'content': 'BM25 is a keyword-based search method useful for exact terms, technical names and categories.', 'published': True, 'topic': 'bm25', 'external_id': 'snap-003', 'title': 'BM25 Keyword Search', 'version': 1}}
--------------------------------------------------------------------------------
{'uuid': '1ed6c91e-2c3a-4eb7-9181-b210c3f116ea', 'properties': {'source': 'notebook', 'content': 'Generative search retrieves relevant objects and sends them to a language model to produce an answer.', 'published': True, 'topic': 'generative_search', 'external_id': 'snap-005', 'title': 'Generative Search', 'version': 1}}
--------------------------------------------------------------------------------
{'uuid': '31948174-9e8e-45cc-8fe2-5760d7097f86', 'properties': {'source': 'notebook', 'title': 'Vector Search', 'published': True, 'topic': 'vector_search', 'external_id': 'snap-002', 'content

In [14]:
TARGET_COLLECTION_NAME = "SnapshotArticleRestored"

if client.collections.exists(TARGET_COLLECTION_NAME):
    client.collections.delete(TARGET_COLLECTION_NAME)

restored_articles = client.collections.create(
    name=TARGET_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="external_id",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="version",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="published",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", TARGET_COLLECTION_NAME)

Created collection: SnapshotArticleRestored


In [15]:
restored_data = [
    record["properties"]
    for record in snapshot_records
]

for item in restored_data[:3]:
    print(item)
    print("-" * 80)

{'source': 'notebook', 'content': 'BM25 is a keyword-based search method useful for exact terms, technical names and categories.', 'published': True, 'topic': 'bm25', 'external_id': 'snap-003', 'title': 'BM25 Keyword Search', 'version': 1}
--------------------------------------------------------------------------------
{'source': 'notebook', 'content': 'Generative search retrieves relevant objects and sends them to a language model to produce an answer.', 'published': True, 'topic': 'generative_search', 'external_id': 'snap-005', 'title': 'Generative Search', 'version': 1}
--------------------------------------------------------------------------------
{'source': 'notebook', 'title': 'Vector Search', 'published': True, 'topic': 'vector_search', 'external_id': 'snap-002', 'content': 'Vector search finds semantically similar objects by comparing embeddings in vector space.', 'version': 1}
--------------------------------------------------------------------------------


In [16]:
result = restored_articles.data.insert_many(restored_data)

if result.errors:
    print("Restore errors:")
    print(result.errors)
else:
    print("Restore completed.")

Restore completed.


In [17]:
source_count = articles.aggregate.over_all(total_count=True).total_count
target_count = restored_articles.aggregate.over_all(total_count=True).total_count

print("Source count:", source_count)
print("Restored count:", target_count)
print("Counts match:", source_count == target_count)

Source count: 10
Restored count: 10
Counts match: True


In [18]:
response = articles.query.near_text(
    query="how to build RAG with notebooks and markdown notes",
    limit=3,
    return_properties=[
        "external_id",
        "title",
        "content",
        "topic",
        "source",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Source:", obj.properties["source"])
    print("-" * 80)

Distance: 0.3139258623123169
Title: Focused RAG
Topic: rag
Source: markdown
--------------------------------------------------------------------------------
Distance: 0.6497060060501099
Title: Generative Search
Topic: generative_search
Source: notebook
--------------------------------------------------------------------------------
Distance: 0.6609953045845032
Title: Cross-References
Topic: cross_references
Source: notebook
--------------------------------------------------------------------------------


In [19]:
response = restored_articles.query.near_text(
    query="how to build RAG with notebooks and markdown notes",
    limit=3,
    return_properties=[
        "external_id",
        "title",
        "content",
        "topic",
        "source",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Source:", obj.properties["source"])
    print("-" * 80)

Distance: 0.37418854236602783
Title: Focused RAG
Topic: rag
Source: markdown
--------------------------------------------------------------------------------
Distance: 0.6663830280303955
Title: Generative Search
Topic: generative_search
Source: notebook
--------------------------------------------------------------------------------
Distance: 0.6731424927711487
Title: Cross-References
Topic: cross_references
Source: notebook
--------------------------------------------------------------------------------


In [20]:
response = restored_articles.query.fetch_objects(
    filters=Filter.by_property("published").equal(True),
    limit=20,
    return_properties=[
        "external_id",
        "title",
        "topic",
        "published",
    ],
)

for obj in response.objects:
    print(obj.properties)

{'published': True, 'topic': 'generative_search', 'external_id': 'snap-005', 'title': 'Generative Search'}
{'published': True, 'topic': 'bm25', 'external_id': 'snap-003', 'title': 'BM25 Keyword Search'}
{'published': True, 'topic': 'rag', 'external_id': 'snap-006', 'title': 'Focused RAG'}
{'published': True, 'topic': 'vector_search', 'external_id': 'snap-002', 'title': 'Vector Search'}
{'published': True, 'topic': 'multi_tenancy', 'external_id': 'snap-009', 'title': 'Multi-Tenancy'}
{'published': True, 'topic': 'data_ingestion', 'external_id': 'snap-010', 'title': 'Idempotent Imports'}
{'published': True, 'topic': 'hybrid_search', 'external_id': 'snap-004', 'title': 'Hybrid Search'}
{'published': True, 'topic': 'named_vectors', 'external_id': 'snap-007', 'title': 'Named Vectors'}
{'published': True, 'topic': 'cloud', 'external_id': 'snap-001', 'title': 'Weaviate Cloud Authentication'}


In [21]:
PUBLISHED_SNAPSHOT_PATH = SNAPSHOT_DIR / "snapshot_articles_published.jsonl"

published_records = []

response = articles.query.fetch_objects(
    filters=Filter.by_property("published").equal(True),
    limit=100,
    return_properties=[
        "external_id",
        "title",
        "content",
        "topic",
        "source",
        "version",
        "published",
    ],
)

for obj in response.objects:
    published_records.append(
        {
            "uuid": str(obj.uuid),
            "properties": obj.properties,
        }
    )

with PUBLISHED_SNAPSHOT_PATH.open("w", encoding="utf-8") as file:
    for record in published_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Published records exported:", len(published_records))
print("Path:", PUBLISHED_SNAPSHOT_PATH)

Published records exported: 9
Path: vector_db/snapshots/snapshot_articles_published.jsonl


In [22]:
def build_snapshot_report(records: list[dict[str, Any]]) -> dict[str, Any]:
    topics = {}
    sources = {}
    published_count = 0

    for record in records:
        props = record["properties"]

        topic = props["topic"]
        source = props["source"]

        topics[topic] = topics.get(topic, 0) + 1
        sources[source] = sources.get(source, 0) + 1

        if props["published"]:
            published_count += 1

    return {
        "total_records": len(records),
        "published_count": published_count,
        "draft_count": len(records) - published_count,
        "topics": topics,
        "sources": sources,
    }

In [23]:
snapshot_report = build_snapshot_report(snapshot_records)

snapshot_report

{'total_records': 10,
 'published_count': 9,
 'draft_count': 1,
 'topics': {'bm25': 1,
  'generative_search': 1,
  'vector_search': 1,
  'rag': 1,
  'cross_references': 1,
  'multi_tenancy': 1,
  'data_ingestion': 1,
  'hybrid_search': 1,
  'named_vectors': 1,
  'cloud': 1},
 'sources': {'notebook': 9, 'markdown': 1}}

In [24]:
response = restored_articles.generate.near_text(
    query="important Weaviate concepts for building practical applications",
    grouped_task=(
        "Summarize the retrieved restored articles. "
        "Explain which Weaviate concepts are represented in this restored snapshot. "
        "Use only the retrieved objects."
    ),
    limit=5,
    return_properties=[
        "external_id",
        "title",
        "content",
        "topic",
        "source",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["title"],
        "|",
        obj.properties["topic"],
        "|",
        obj.properties["source"],
    )

The retrieved articles provide insights into various Weaviate concepts related to its functionalities, particularly focusing on authentication, search methods, vector representation, and object connections.

1. **Weaviate Cloud Authentication**: This article outlines how Weaviate Cloud uses a cluster URL and API key for authentication. It also mentions the use of provider headers, such as the OpenAI API key, for vectorization and content generation.

2. **Vector Search**: The focus of this article is on vector search, which identifies semantically similar objects by analyzing their embeddings in vector space. This concept underlines Weaviate's capability to perform semantic searches effectively.

3. **Named Vectors**: This article explains the use of named vectors, allowing a single object to have multiple vector representations, such as ‘title_vector’ and ‘description_vector.’ This facilitates more nuanced search and retrieval processes by differentiating contexts within the same obje

In [25]:
client.close()